# LOF Modeling — Amir (v2)

Tahap modeling **Local Outlier Factor (LOF)** sepenuhnya *label-free* dalam pemilihan hyperparameter, memperbaiki masalah label-guided tuning pada versi lama.

Mengikuti spec: `docs/superpowers/specs/2026-09-12-lof-redo-design.md` §5.

**Prinsip utama:** `crisis_label` TIDAK PERNAH dipakai untuk memilih `n_neighbors`, aturan agregasi, atau threshold anomali — hanya dipakai di tahap evaluasi (setelah skor/prediksi selesai dihitung). Ini menempatkan LOF pada level yang sama dengan DBSCAN dalam hal seberapa banyak informasi label yang "bocor" ke model, menjawab langsung kritik metodologis di laporan lama.

**Catatan penting:** Metodologi ini (label-free tuning, `RobustScaler` di tahap preprocessing) sengaja menyimpang dari proposal final kelompok Bab III §3.2.1 (`StandardScaler` untuk semua algoritma) dan §3.2.3 (grid search `n_neighbors` via AUC-ROC terhadap `crisis_label`). Proposal akan direvisi menyusul; ini bukan penyimpangan diam-diam.

## 1. Load Data & Hitung Skor LOF per-k (Transductive, Label-Free)

In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import LocalOutlierFactor

df = pd.read_csv('data_cleaned_v2.csv')
feature_cols = [c for c in df.columns if c not in ('economy', 'year', 'crisis_label')]
X = df[feature_cols].values
y = df['crisis_label'].values

k_range = [5, 10, 15, 20, 30, 50]
scores = np.zeros((len(df), len(k_range)))
for i, k in enumerate(k_range):
    lof = LocalOutlierFactor(n_neighbors=k, metric='euclidean', novelty=False)
    lof.fit_predict(X)
    scores[:, i] = -lof.negative_outlier_factor_

print(f'Fitur dipakai ({len(feature_cols)}): {feature_cols}')
print(f'crisis_label TIDAK ada di X: {"crisis_label" not in feature_cols}')
print(f'Shape scores: {scores.shape}')
for i, k in enumerate(k_range):
    print(f'k={k}: mean={scores[:, i].mean():.3f} min={scores[:, i].min():.3f} max={scores[:, i].max():.3f}')

Fitur dipakai (14): ['GDP_Growth', 'Inflation_CPI', 'Unemployment', 'Current_Account_GDP', 'Reserves_Months_Imports', 'FDI_Inflows_GDP', 'Exports_GDP', 'Imports_GDP', 'Gross_Savings_GDP', 'Investment_GDP', 'Manufacturing_Value', 'Domestic_Credit_GDP', 'Broad_Money_Growth', 'Exchange_Depreciation']
crisis_label TIDAK ada di X: True
Shape scores: (1715, 6)
k=5: mean=1.146 min=0.913 max=12.953
k=10: mean=1.151 min=0.947 max=13.307
k=15: mean=1.172 min=0.957 max=16.871
k=20: mean=1.201 min=0.962 max=24.861
k=30: mean=1.269 min=0.961 max=59.355
k=50: mean=1.362 min=0.957 max=103.293


## 2. Agregasi Max-over-k, Threshold, Simpan `hasil_lof_v2.csv`

`n_neighbors` tidak dipilih satu nilai lewat label — dipakai rentang `{5,10,15,20,30,50}` (sama seperti rentang grid search di proposal final Bab III, tapi diagregasi lewat MAX per Breunig et al. 2000, bukan diseleksi via AUC-ROC terhadap label). Threshold `1.5` adalah aturan `contamination='auto'` bawaan sklearn, bukan diturunkan dari rasio krisis.

In [2]:
df['anomaly_score'] = scores.max(axis=1)
df['predicted_anomaly'] = (df['anomaly_score'] > 1.5).astype(int)

hasil = df[['economy', 'year', 'anomaly_score', 'predicted_anomaly']]
assert list(hasil.columns) == ['economy', 'year', 'anomaly_score', 'predicted_anomaly']
assert len(hasil) == 1715
hasil.to_csv('hasil_lof_v2.csv', index=False)

print(f'anomaly_score: min={df["anomaly_score"].min():.4f} max={df["anomaly_score"].max():.4f} mean={df["anomaly_score"].mean():.4f}')
print(f'predicted_anomaly: {int(df["predicted_anomaly"].sum())} baris ({df["predicted_anomaly"].mean():.1%})')
print('Disimpan ke hasil_lof_v2.csv dengan kolom:', list(hasil.columns))

anomaly_score: min=0.9785 max=103.2925 mean=1.4187
predicted_anomaly: 171 baris (10.0%)
Disimpan ke hasil_lof_v2.csv dengan kolom: ['economy', 'year', 'anomaly_score', 'predicted_anomaly']


## 3. Evaluasi Utama terhadap `crisis_label`

Di sinilah SATU-SATUNYA tempat `crisis_label` dipakai di seluruh notebook ini — murni untuk mengukur kualitas skor yang sudah dihitung tanpa label, bukan untuk menentukan skor itu sendiri.

In [3]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, confusion_matrix,
)

y = df['crisis_label'].values
auc = roc_auc_score(y, df['anomaly_score'])
ap = average_precision_score(y, df['anomaly_score'])
prec = precision_score(y, df['predicted_anomaly'])
rec = recall_score(y, df['predicted_anomaly'])
f1 = f1_score(y, df['predicted_anomaly'])
cm = confusion_matrix(y, df['predicted_anomaly'])

print(f'AUC-ROC   = {auc:.4f}')
print(f'AP        = {ap:.4f}')
print(f'Precision = {prec:.4f}')
print(f'Recall    = {rec:.4f}')
print(f'F1-Score  = {f1:.4f}')
print('Confusion matrix [[TN, FP], [FN, TP]]:')
print(cm)

gt = pd.read_csv('ground_truth_imf.csv')[['economy', 'year', 'crisis_name', 'is_crisis']]
merged = df.merge(gt, on=['economy', 'year'], how='left')
per_crisis = merged[merged['is_crisis'] == 1].groupby('crisis_name').agg(
    observasi=('is_crisis', 'size'),
    terdeteksi=('predicted_anomaly', 'sum'),
).reset_index()
per_crisis['detection_rate_pct'] = (per_crisis['terdeteksi'] / per_crisis['observasi'] * 100).round(1)
per_crisis = per_crisis.sort_values('observasi', ascending=False)
n_unnamed = int(merged[(merged['is_crisis'] == 1) & (merged['crisis_name'].isna())].shape[0])
print(f'\nPer-crisis breakdown ({len(per_crisis)} episode bernama, {n_unnamed} baris krisis tanpa nama episode):')
print(per_crisis.to_string(index=False))

top10 = df.groupby('economy')['predicted_anomaly'].sum().sort_values(ascending=False).head(10)
print('\nTop 10 negara dengan anomali terbanyak:')
print(top10.to_string())

AUC-ROC   = 0.7099
AP        = 0.2427
Precision = 0.2456
Recall    = 0.1834
F1-Score  = 0.2100
Confusion matrix [[TN, FP], [FN, TP]]:
[[1357  129]
 [ 187   42]]

Per-crisis breakdown (28 episode bernama, 22 baris krisis tanpa nama episode):
                          crisis_name  observasi  terdeteksi  detection_rate_pct
                COVID-19 Global Shock         49          14                28.6
              Global Financial Crisis         47           3                 6.4
               Asian Financial Crisis         19           6                31.6
              Nigerian Banking Crisis          9           2                22.2
            Transition Banking Crisis          8           0                 0.0
         GFC & Spanish Banking Crisis          5           0                 0.0
          Greek Debt & Banking Crisis          5           1                20.0
             Real Plan Banking Crisis          5           1                20.0
                 Irish Banking

## 4. Robustness Check — Stratified Resampling (`random_state=42`)

`crisis_label` dipakai lagi di sini, tapi HANYA untuk stratifikasi sampling dan penilaian akhir tiap subsample — bukan untuk memilih parameter. Mengulang prosedur label-free yang identik (Task 1-2) pada 20 subsample stratified 80%, untuk menunjukkan hasil utama bukan hasil kebetulan satu kali fit.

In [4]:
from sklearn.model_selection import StratifiedShuffleSplit

def lof_max_score(X_sub):
    sub_scores = np.zeros((len(X_sub), len(k_range)))
    for i, k in enumerate(k_range):
        k_eff = min(k, len(X_sub) - 1)
        lof = LocalOutlierFactor(n_neighbors=k_eff, metric='euclidean', novelty=False)
        lof.fit_predict(X_sub)
        sub_scores[:, i] = -lof.negative_outlier_factor_
    return sub_scores.max(axis=1)

sss = StratifiedShuffleSplit(n_splits=20, train_size=0.8, random_state=42)
aucs, f1s = [], []
for train_idx, _ in sss.split(X, y):
    Xs, ys = X[train_idx], y[train_idx]
    s = lof_max_score(Xs)
    pred = (s > 1.5).astype(int)
    aucs.append(roc_auc_score(ys, s))
    f1s.append(f1_score(ys, pred))

aucs = np.array(aucs)
f1s = np.array(f1s)
print(f'AUC-ROC: mean={aucs.mean():.4f} std={aucs.std():.4f} min={aucs.min():.4f} max={aucs.max():.4f}')
print(f'F1     : mean={f1s.mean():.4f} std={f1s.std():.4f} min={f1s.min():.4f} max={f1s.max():.4f}')

AUC-ROC: mean=0.7068 std=0.0106 min=0.6846 max=0.7264
F1     : mean=0.2265 std=0.0153 min=0.2063 max=0.2620


## 5. Simpan `grid_search_lof_v2.csv` dan `ringkasan_model_lof_v2.csv`

In [5]:
grid_search = pd.DataFrame({
    'n_neighbors': k_range,
    'mean_lof_score': scores.mean(axis=0),
    'pct_score_over_1.5': (scores > 1.5).mean(axis=0) * 100,
})
grid_search.to_csv('grid_search_lof_v2.csv', index=False)
print(grid_search.to_string(index=False))

ringkasan = pd.DataFrame([{
    'metode': 'Local Outlier Factor (label-free)',
    'n_neighbors_range': str(k_range),
    'aggregation': 'max LOF score across n_neighbors range (Breunig et al. 2000)',
    'contamination_rule': 'anomaly_score > 1.5 (sklearn contamination=auto equivalent)',
    'label_used_in_tuning': False,
    'label_used_in_threshold': False,
    'random_state': 42,
    'n_rows': len(df),
    'n_predicted_anomaly': int(df['predicted_anomaly'].sum()),
    'auc_roc_primary': round(auc, 4),
    'average_precision_primary': round(ap, 4),
    'precision_primary': round(prec, 4),
    'recall_primary': round(rec, 4),
    'f1_primary': round(f1, 4),
    'auc_roc_resample_mean': round(aucs.mean(), 4),
    'auc_roc_resample_std': round(aucs.std(), 4),
    'f1_resample_mean': round(f1s.mean(), 4),
    'f1_resample_std': round(f1s.std(), 4),
}])
ringkasan.to_csv('ringkasan_model_lof_v2.csv', index=False)
print()
print(ringkasan.T.to_string())

 n_neighbors  mean_lof_score  pct_score_over_1.5
           5        1.145591            5.072886
          10        1.151250            4.839650
          15        1.171923            5.189504
          20        1.200617            5.131195
          30        1.268562            6.122449
          50        1.361789            7.405248

                                                                                      0
metode                                                Local Outlier Factor (label-free)
n_neighbors_range                                               [5, 10, 15, 20, 30, 50]
aggregation                max LOF score across n_neighbors range (Breunig et al. 2000)
contamination_rule          anomaly_score > 1.5 (sklearn contamination=auto equivalent)
label_used_in_tuning                                                              False
label_used_in_threshold                                                           False
random_state                            

## Ringkasan

| Tahap | Hasil |
|---|---|
| `n_neighbors` | Rentang `{5,10,15,20,30,50}`, diagregasi via MAX (Breunig et al. 2000) — tanpa label |
| Threshold | `anomaly_score > 1.5` (`contamination='auto'` sklearn) — tanpa label |
| Evaluasi utama | AUC-ROC 0,7099; F1 0,2100 — diukur SETELAH skor selesai, bukan dipakai memilih parameter |
| Robustness (20 resample) | AUC-ROC 0,7068 ± 0,0106; F1 0,2265 ± 0,0153 — sebaran ketat, hasil utama bukan kebetulan |

**Perbandingan jujur dengan versi lama:** F1 lama (0,3435) lebih tinggi tapi diukur dari `AUC-ROC` yang di-tuning pada label yang sama dengan yang dilaporkan — angka itu optimis palsu. F1 versi ini (0,2100 primer / 0,2265±0,0153 resampled) lebih rendah tapi jujur, mengukur generalisasi, bukan seberapa pas parameter di-fit ke label.

**Output:** `Amir/hasil_lof_v2.csv`, `Amir/grid_search_lof_v2.csv`, `Amir/ringkasan_model_lof_v2.csv` — siap untuk `laporan_lof_v2.md` dan tahap majority voting kelompok.